# ODLS ablation: weight sharing only -- Colab runner

Runs `odls_ablation_sharing/` -- isolates weight sharing across phases
(one shared network per role, conditioned on a learnable phase
embedding) from `odls_v2`'s other two changes (the denoiser reframing,
cross-phase attention). See `odls_ablation_sharing/README.md` for why
this exists. Its own separate `CHECKPOINT_DIR`, so it never collides
with the baseline or `odls_v2`'s checkpoints.

Same data as the other notebooks -- if you've already staged it in this
Colab session, this reuses the local copies rather than re-downloading.

Edit the **Config** cell below to match your Drive folder paths, then
run all cells top to bottom.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Config -- edit these paths

In [ ]:
DRIVE_DATA_DIR = "/content/drive/MyDrive/odls_project/data/raw_corpd"
DRIVE_TEST_DATA_DIR = "/content/drive/MyDrive/odls_project/data/test_corpd"

LOCAL_RAW_DIR = "/content/fastmri_raw"
LOCAL_TRAIN_DIR = "/content/fastmri_corpd/train"
LOCAL_VAL_DIR = "/content/fastmri_corpd/val"
LOCAL_TEST_RAW_DIR = "/content/fastmri_test_raw"
LOCAL_TEST_DIR = "/content/fastmri_corpd/test"

REPO_URL = "https://github.com/Shambhawi419/1D_MRI.git"
REPO_DIR = "/content/1D_MRI"

# Separate from every other notebook's checkpoint folder -- a different
# model (ODLSAblationSharing), and this experiment specifically needs to
# be compared against ../odls's baseline and odls_v2's combined model
# without any of the three ever overwriting each other.
CHECKPOINT_DIR = "/content/drive/MyDrive/odls_ablation_sharing_checkpoints"

VAL_FRACTION = 0.2
SPLIT_SEED = 0

N_COILS = 8
N_VIRTUAL_COILS = 8
MASK_TYPE = "cartesian"
AF = 4.0

CROP_FE_TO = 224
CROP_PE_TO = 224

MAX_THRESHOLD = 0.05
EMBED_DIM = 16
WIDTH = 48   # paper-matching default -- the width=64 experiment on the
             # full odls_v2 model underperformed, so there's no reason
             # to start this ablation anywhere but the original width

## Stage the data locally

In [ ]:
import os
import shutil
import tarfile
import zipfile


def stage_files(drive_dir, local_dir):
    os.makedirs(local_dir, exist_ok=True)

    if not os.path.isdir(drive_dir):
        raise FileNotFoundError(
            f"{drive_dir} not found -- check the Drive is mounted and the "
            "folder path/name is correct."
        )

    n_copied, n_extracted = 0, 0
    for name in sorted(os.listdir(drive_dir)):
        src = os.path.join(drive_dir, name)
        if os.path.isdir(src):
            continue

        if name.lower().endswith((".h5", ".hdf5")):
            dst = os.path.join(local_dir, name)
            if not os.path.exists(dst):
                shutil.copy2(src, dst)
            n_copied += 1

        elif zipfile.is_zipfile(src):
            with zipfile.ZipFile(src) as zf:
                zf.extractall(local_dir)
            n_extracted += 1

        elif tarfile.is_tarfile(src):
            with tarfile.open(src) as tf:
                tf.extractall(local_dir)
            n_extracted += 1

    for root, _, files in os.walk(local_dir):
        if root == local_dir:
            continue
        for f in files:
            if f.lower().endswith((".h5", ".hdf5")):
                src = os.path.join(root, f)
                dst = os.path.join(local_dir, f)
                if not os.path.exists(dst):
                    shutil.move(src, dst)

    h5_files = sorted(
        f for f in os.listdir(local_dir) if f.lower().endswith((".h5", ".hdf5"))
    )
    print(f"{drive_dir}: copied {n_copied} loose .h5 files, extracted {n_extracted} archive(s)")
    print(f"{len(h5_files)} .h5 files staged locally in {local_dir}")
    return h5_files


h5_files = stage_files(DRIVE_DATA_DIR, LOCAL_RAW_DIR)
test_h5_files = stage_files(DRIVE_TEST_DATA_DIR, LOCAL_TEST_RAW_DIR)

## Clone the repo and filter to CORPD_FBK

In [ ]:
import subprocess

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

import sys
sys.path.insert(0, os.path.join(REPO_DIR, "odls_ablation_sharing"))

from fastmri_data import find_corpd_files

corpd_files = find_corpd_files(LOCAL_RAW_DIR, fat_suppressed=False)
print(f"train/val pool: {len(corpd_files)} of {len(h5_files)} staged files are CORPD_FBK")
for p in corpd_files:
    print(" ", os.path.basename(p))

if not corpd_files:
    raise RuntimeError(
        "No CORPD_FBK files found among the staged train/val data."
    )

test_corpd_files = find_corpd_files(LOCAL_TEST_RAW_DIR, fat_suppressed=False)
print(f"\ntest pool: {len(test_corpd_files)} of {len(test_h5_files)} staged files are CORPD_FBK")
for p in test_corpd_files:
    print(" ", os.path.basename(p))

if not test_corpd_files:
    raise RuntimeError(
        "No CORPD_FBK files found among the staged test data."
    )

## Split into train/val/test folders

In [ ]:
import random

os.makedirs(LOCAL_TRAIN_DIR, exist_ok=True)
os.makedirs(LOCAL_VAL_DIR, exist_ok=True)
os.makedirs(LOCAL_TEST_DIR, exist_ok=True)

rng = random.Random(SPLIT_SEED)
shuffled = corpd_files[:]
rng.shuffle(shuffled)

n_val = max(1, int(round(len(shuffled) * VAL_FRACTION))) if len(shuffled) > 1 else 0
val_files = shuffled[:n_val]
train_files = shuffled[n_val:]

def _relink(files, dest_dir):
    for src in files:
        link_path = os.path.join(dest_dir, os.path.basename(src))
        if os.path.lexists(link_path):
            os.remove(link_path)
        os.symlink(src, link_path)

_relink(train_files, LOCAL_TRAIN_DIR)
_relink(val_files, LOCAL_VAL_DIR)
_relink(test_corpd_files, LOCAL_TEST_DIR)

print(f"train: {len(train_files)} files -> {LOCAL_TRAIN_DIR}")
print(f"val:   {len(val_files)} files -> {LOCAL_VAL_DIR}")
print(f"test:  {len(test_corpd_files)} files -> {LOCAL_TEST_DIR}")

## Train (weight-sharing-only ablation)

Same auto-resume behavior as the other notebooks: if this cell's runtime
disconnects, re-run it and it picks up from the last completed epoch.

In [ ]:
!cd {REPO_DIR}/odls_ablation_sharing && python train.py \
    --fastmri-train-root {LOCAL_TRAIN_DIR} \
    --fastmri-val-root {LOCAL_VAL_DIR} \
    --n-coils {N_COILS} --n-virtual-coils {N_VIRTUAL_COILS} \
    --crop-fe-to {CROP_FE_TO} --crop-pe-to {CROP_PE_TO} \
    --max-threshold {MAX_THRESHOLD} --embed-dim {EMBED_DIM} --width {WIDTH} \
    --mask-type {MASK_TYPE} --af {AF} \
    --num-workers 2 \
    --checkpoint-dir {CHECKPOINT_DIR}

## Test (weight-sharing-only ablation)

In [ ]:
!cd {REPO_DIR}/odls_ablation_sharing && python evaluate.py \
    --fastmri-test-root {LOCAL_TEST_DIR} \
    --checkpoint {CHECKPOINT_DIR}/odls_best.pt \
    --n-coils {N_COILS} --n-virtual-coils {N_VIRTUAL_COILS} \
    --crop-fe-to {CROP_FE_TO} --crop-pe-to {CROP_PE_TO} \
    --max-threshold {MAX_THRESHOLD} --embed-dim {EMBED_DIM} --width {WIDTH} \
    --mask-type {MASK_TYPE} --af {AF}